# ADNI Data Preprocessing Pipeline — Step-by-Step Demo

This notebook demonstrates the preprocessing pipeline used to prepare data from the **Alzheimer's Disease Neuroimaging Initiative (ADNI)** for downstream analysis.

We use the `ADNIPreprocess` class to walk through each stage of the pipeline on a **small representative subset** of the data, showing intermediate results and explaining _what_ each step does and _why_.

### Pipeline Overview

| Step | Method | Purpose |
|------|--------|---------|
| 1 | `load_data()` | Load the raw clinical and MRI CSV files |
| 2 | `encode_multihot_variables()` | One-hot encode pipe-delimited multi-response columns |
| 3 | `coerce_numeric_columns()` | Fix string-contaminated numeric columns |
| 4 | `compute_bmi()` | Derive standardized height, weight, and BMI |
| 5 | `filter_cn_subjects()` | Identify CN subjects with single vs. multiple diagnoses |
| 6 | `build_transition_cohort()` | Extract CN→MCI/AD transition cohort |
| 7 | `build_no_transition_cohort()` | Extract stable CN cohort |
| 8 | `match_cohorts()` | Propensity-match transition ↔ no-transition subjects |
| 9 | `select_features()` | Remove near-zero-variance features |
| 10 | `export_datasets()` | Save final datasets to CSV |

---
## Setup

Import the `ADNIPreprocess` class and configure logging so we can see informational messages from each pipeline step.

In [ ]:
import logging
import pandas as pd
import numpy as np

from src.data_preprocessing import ADNIPreprocess

logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')

# Instantiate the preprocessor pointing to the demo subset
preprocessor = ADNIPreprocess(
    data_path="data/demo_subjects.csv",
    mri_path="data/demo_mri.csv",
    output_dir="data/demo_output/",
    variance_threshold=0.01,
)
print("ADNIPreprocess instance created.")

---
## Step 1 — Load Data

The first step loads two CSV files:

- **Subjects table** (`data_df`): Contains clinical, demographic, cognitive, and biomarker variables for each subject at every visit.
- **MRI table** (`mri_df`): Contains FreeSurfer-derived structural MRI measurements (cortical thickness, volumes, etc.).

Each subject has **multiple rows** — one per study visit (screening, baseline, follow-up at 6, 12, 24, … months).

In [ ]:
preprocessor.load_data()

print(f"Subjects table: {preprocessor.data_df.shape[0]} rows × {preprocessor.data_df.shape[1]} columns")
print(f"MRI table:      {preprocessor.mri_df.shape[0]} rows × {preprocessor.mri_df.shape[1]} columns")
print(f"\nUnique subjects: {preprocessor.data_df['subject_id'].nunique()}")
print(f"Unique visits:   {preprocessor.data_df['visit'].unique()}")

In [ ]:
# Preview the first few rows of the subjects table
preprocessor.data_df.head()

---
## Step 2 — Multi-Hot Encoding

Several ADNI columns store **multiple responses in a single cell** using pipe-delimited values. For example, the `KEYMED` column (key medications) might contain `"1|4|6"` meaning the patient is on Aricept (1), Namenda (4), and anti-depressants (6).

This step **one-hot encodes** each of these columns, expanding them into separate binary columns:

| Original `KEYMED` | → `KEYMED_0` | `KEYMED_1` | `KEYMED_4` | `KEYMED_6` | ... |
|:------------------:|:---:|:---:|:---:|:---:|:---:|
| `"1\|4\|6"`           | 0   | 1   | 1   | 1   | ... |
| `"0"`               | 1   | 0   | 0   | 0   | ... |
| `NaN`              | NaN | NaN | NaN | NaN | ... |

**Encoded variables:** `DXMOTHET`, `KEYMED`, `PTNLANG`, `MHNUM`, `PTMARRY`, `PTHOME`, `PTNOTRT`, `PTPLANG`

In [ ]:
n_cols_before = preprocessor.data_df.shape[1]

# Show a sample of the raw KEYMED column before encoding
print("Before encoding — sample KEYMED values:")
print(preprocessor.data_df['KEYMED'].dropna().head(10).tolist())

preprocessor.encode_multihot_variables()

n_cols_after = preprocessor.data_df.shape[1]
print(f"\nColumns before: {n_cols_before}  →  after: {n_cols_after}  (+{n_cols_after - n_cols_before} new one-hot columns)")

In [ ]:
# Show the newly created KEYMED one-hot columns
keymed_cols = [c for c in preprocessor.data_df.columns if c.startswith('KEYMED_')]
print(f"KEYMED expanded into {len(keymed_cols)} binary columns: {keymed_cols}")
preprocessor.data_df[['KEYMED'] + keymed_cols].dropna().head()

---
## Step 3 — Numeric Coercion

Some columns that should be numeric contain **string artifacts** (e.g., `">1700"` or `"<5"` in lab results). This step forces those columns to numeric, converting unparseable strings to `NaN`.

Additionally, the **CV** (Coefficient of Variation) column contains values like `"12.5%"` — the trailing `%` is stripped and the result converted to float.

**Affected columns:** `RCT20`, `RCT1408`, `RCT19`, `HMT40`, `BAT126`, `RCT392`, `RCT14`, `CV`

In [ ]:
# Show sample CV values before coercion
print("Before coercion — sample CV values:")
print(preprocessor.data_df['CV'].dropna().head(5).tolist())

print("\nBefore coercion — sample RCT20 values:")
print(preprocessor.data_df['RCT20'].dropna().head(5).tolist())

preprocessor.coerce_numeric_columns()

print("\nAfter coercion — sample CV values (float):")
print(preprocessor.data_df['CV'].dropna().head(5).tolist())

print("\nAfter coercion — sample RCT20 values (float):")
print(preprocessor.data_df['RCT20'].dropna().head(5).tolist())

---
## Step 4 — BMI Computation

ADNI records height and weight in **mixed units** (imperial or metric), indicated by unit columns (`VSHTUNIT`, `VSWTUNIT`).

This step:
1. Converts all heights to **centimeters** (inches × 2.54 if imperial)
2. Converts all weights to **kilograms** (lbs × 0.4536 if imperial)
3. Computes **BMI** = weight(kg) / (height(m))²

Three new columns are added: `HEIGHT`, `WEIGHT`, `BMI`.

In [ ]:
preprocessor.compute_bmi()

bmi_cols = ['VSHEIGHT', 'VSHTUNIT', 'HEIGHT', 'VSWEIGHT', 'VSWTUNIT', 'WEIGHT', 'BMI']
preprocessor.data_df[bmi_cols].dropna().head(10)

In [ ]:
# Summary statistics for BMI
print("BMI distribution:")
preprocessor.data_df['BMI'].describe()

---
## Step 5 — Filter Cognitively Normal (CN) Subjects

We focus on subjects whose **initial research group** was **Cognitively Normal (CN)**.

These CN subjects are split into two groups based on their longitudinal diagnostic trajectory:

| Group | Description | Clinical meaning |
|-------|-------------|------------------|
| **Multiple diagnoses** | Subject received ≥2 distinct diagnoses across visits | Potentially transitioned from CN → MCI or CN → Dementia |
| **Single diagnosis** | Subject always received the same diagnosis | Remained cognitively stable throughout the study |

This classification is the first step toward building the **transition** and **no-transition** cohorts.

In [ ]:
preprocessor.filter_cn_subjects()

print(f"CN subjects with MULTIPLE diagnoses: {len(preprocessor._subjects_multiple_dx)}")
print(f"CN subjects with SINGLE diagnosis:   {len(preprocessor._subjects_one_dx)}")

In [ ]:
# Example: show the diagnostic trajectory of a multi-diagnosis CN subject
example_subj = preprocessor._subjects_multiple_dx[0]
subj_df = preprocessor.data_df[preprocessor.data_df['subject_id'] == example_subj]
print(f"Subject: {example_subj}")
print(f"Research group: {subj_df['research_group'].iloc[0]}")
subj_df[['visit', 'DIAGNOSIS']].dropna()

---
## Step 6 — Build the Transition Cohort

From the CN subjects with multiple diagnoses, we apply stricter **inclusion criteria** to build the _transition cohort_:

1. The subject must have **remained CN for at least the first 12 months** of the study — this ensures the baseline measurements were taken while the subject was still cognitively normal.
2. The subject's **last recorded diagnosis must be MCI (2) or Dementia (3)** — confirming an actual cognitive decline.

For each qualifying subject, we extract the **baseline measurements** (screening + baseline visits, back-filled/forward-filled to handle missing data) along with:
- `study_duration`: total months of follow-up
- `last_diagnosis`: the final diagnosis (MCI=2 or Dementia=3)
- `transition`: set to **1** (indicating this subject transitioned)

In [ ]:
preprocessor.build_transition_cohort()

print(f"Transition cohort size: {preprocessor.subjects_transition_df.shape[0]} subjects")
print(f"\nLast diagnosis distribution (2=MCI, 3=Dementia):")
print(preprocessor.subjects_transition_df['last_diagnosis'].value_counts())
print(f"\nStudy duration (months) — summary:")
print(preprocessor.subjects_transition_df['study_duration'].describe())

In [ ]:
# Preview transition cohort
preprocessor.subjects_transition_df[[
    'subject_id', 'subject_age', 'PTGENDER', 'GENOTYPE',
    'DIAGNOSIS', 'last_diagnosis', 'study_duration', 'transition'
]]

---
## Step 7 — Build the No-Transition Cohort

From CN subjects with a **single diagnosis** throughout all visits, we extract the same baseline information.

These subjects serve as the **control pool** — they remained cognitively normal for the entire study period. We extract more subjects than needed here because the next step (matching) will select the best matches and leave the rest as a separate test set.

Each subject gets:
- `study_duration`: total months of follow-up
- `last_diagnosis`: confirming CN (1)
- `transition`: set to **0**

In [ ]:
preprocessor.build_no_transition_cohort()

print(f"No-transition cohort size: {preprocessor.subjects_no_transition_df.shape[0]} subjects")
print(f"\nAll have diagnosis = 1 (CN)? {(preprocessor.subjects_no_transition_df['last_diagnosis'] == 1).all()}")

In [ ]:
# Preview no-transition cohort
preprocessor.subjects_no_transition_df[[
    'subject_id', 'subject_age', 'PTGENDER', 'GENOTYPE',
    'DIAGNOSIS', 'last_diagnosis', 'study_duration', 'transition'
]]

---
## Step 8 — Cohort Matching

To create a **balanced case-control study**, we pair each transition subject with the **best-matching** control (no-transition) subject.

### Matching criteria

| Criterion | Type | Strategy |
|-----------|------|----------|
| **Age** (`subject_age`) | Numeric | Minimize Euclidean distance |
| **Sex** (`PTGENDER`) | Categorical | Exact match required |
| **ApoE Genotype** (`GENOTYPE`) | Categorical | Exact match required |

### Algorithm
1. For each transition subject, compute the Euclidean distance (on `subject_age`) to all remaining controls.
2. Filter to only controls with **matching sex and genotype**.
3. Select the **closest** match.
4. Remove the matched control from the pool (matching **without replacement**).

The result is:
- **`joint_dataset_df`**: the paired dataset (transition + matched controls), with a `group` column linking each pair.
- **`remaining_test_df`**: unmatched controls, usable as a held-out test set.

In [ ]:
preprocessor.match_cohorts()

print(f"Joint dataset:     {preprocessor.joint_dataset_df.shape[0]} subjects ({preprocessor.joint_dataset_df.shape[0]//2} matched pairs)")
print(f"Remaining test:    {preprocessor.remaining_test_df.shape[0]} unmatched controls")
print(f"\nTransition distribution in joint dataset:")
print(preprocessor.joint_dataset_df['transition'].value_counts())

In [ ]:
# Show the matched pairs side by side
match_cols = ['subject_id', 'subject_age', 'PTGENDER', 'GENOTYPE', 'transition', 'group']
pairs_df = preprocessor.joint_dataset_df[match_cols].sort_values(by=['group', 'transition'])
pairs_df

In [ ]:
# Verify matching quality: age difference within each pair
transition_ages = preprocessor.subjects_transition_df.set_index('group')['subject_age'].rename('age_transition')
control_ages = preprocessor.subjects_pairs_df.set_index('group')['subject_age'].rename('age_control')
pair_comparison = pd.concat([transition_ages, control_ages], axis=1)
pair_comparison['age_diff'] = (pair_comparison['age_transition'] - pair_comparison['age_control']).abs()
print("Age difference per matched pair:")
print(pair_comparison)
print(f"\nMean age difference: {pair_comparison['age_diff'].mean():.2f} years")

---
## Step 9 — Feature Selection

With all subjects assembled, we apply **variance-based feature selection** using scikit-learn's `VarianceThreshold`.

Features with variance below the threshold are removed — these are columns that are nearly constant across all subjects and carry little discriminative information.

**Categorical/metadata columns** (e.g., `subject_id`, `GENOTYPE`, `visit`) are excluded from this filter and always retained.

In [ ]:
n_before = preprocessor.joint_dataset_df.shape[1]

preprocessor.select_features()

n_after = preprocessor.joint_dataset_df.shape[1]
print(f"Features before selection: {n_before}")
print(f"Features after selection:  {n_after}")
print(f"Removed: {n_before - n_after} near-zero-variance features")
print(f"\nVariance threshold used: {preprocessor.variance_threshold}")

In [ ]:
# Show kept feature names
print(f"Retained features ({len(preprocessor.keep_features)}):")
print(sorted(preprocessor.keep_features))

---
## Step 10 — Export Datasets

The final step saves three output CSV files:

| File | Contents |
|------|----------|
| `joint_dataset.csv` | Matched transition + control pairs (the main analysis dataset) |
| `remaining_test.csv` | Unmatched control subjects (held-out test set) |
| `mri_joint_dataset.csv` | MRI measurements for subjects in the joint dataset |

In [ ]:
preprocessor.export_datasets()

print("\nDone! Output files saved to:", preprocessor.output_dir)

---
## Summary

The `ADNIPreprocess` pipeline transforms raw ADNI data into a clean, analysis-ready dataset through the following stages:

```
Raw CSVs
 │
 ├─ 1. Load data  ──────────────────  2 CSV files → DataFrames
 ├─ 2. Multi-hot encoding  ─────────  Pipe-delimited → binary columns
 ├─ 3. Numeric coercion  ───────────  Fix string artifacts in lab values
 ├─ 4. BMI computation  ────────────  Standardize units → derive BMI
 ├─ 5. Filter CN subjects  ─────────  Split by diagnostic trajectory
 ├─ 6. Transition cohort  ──────────  CN→MCI/AD with 12-month rule
 ├─ 7. No-transition cohort  ───────  Stable CN controls
 ├─ 8. Cohort matching  ────────────  1:1 age/sex/genotype matching
 ├─ 9. Feature selection  ──────────  Remove near-constant features
 └─ 10. Export  ────────────────────  3 output CSVs
```

All steps are encapsulated in the `ADNIPreprocess` class and can be run end-to-end via `preprocessor.run()`, or step-by-step as shown in this notebook.